# SLM Fine-Tuning on Google Colab — Motor OD Document Verification

Train **SFT + QLoRA** on Indian motor insurance claim packs (`verify_docs_sft_500.jsonl`):
extract/verify VIN, plate, estimate, damage enums, claim route, and doc authenticity as compact JSON.

**Runtime → Change runtime type → GPU** (T4 / L4 / A100).

| Environment | Backend |
|-------------|--------|
| Mac (local) | Unsloth MLX |
| Colab (this notebook) | Unsloth CUDA + TRL SFTTrainer |

**Dataset:** 400 train / 50 val / 50 held-out demo (from `data/verify_docs_sft_500.jsonl`).

## 1) Check GPU

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Enable a GPU runtime: Runtime → Change runtime type → GPU"
print("CUDA:", torch.cuda.get_device_name(0))

## 2) Get the project onto Colab

Pick **one** option below.

### Option A — Clone from GitHub
Set `REPO_URL` to your remote (public or with token).

In [ ]:
import os
from pathlib import Path

# >>> EDIT THIS <<<
REPO_URL = ""  # e.g. https://github.com/<you>/slm-finetuning.git
BRANCH = "main"
PROJECT_DIR = Path("/content/SLM-Finetuning")

if REPO_URL:
    if PROJECT_DIR.exists():
        %cd {PROJECT_DIR}
        !git pull
    else:
        !git clone -b {BRANCH} {REPO_URL} {PROJECT_DIR}
        %cd {PROJECT_DIR}
else:
    print("REPO_URL empty — use Option B (Drive) or Option C (upload zip).")

### Option B — Google Drive folder
Upload the project to Drive, then set `DRIVE_PROJECT_PATH`.

In [ ]:
from pathlib import Path

# >>> EDIT THIS <<<
USE_DRIVE = False
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/SLM Finetuning"

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = Path(DRIVE_PROJECT_PATH)
    assert PROJECT_DIR.exists(), f"Not found: {PROJECT_DIR}"
    %cd {PROJECT_DIR}
    print("Using Drive project:", PROJECT_DIR)
else:
    print("Drive option skipped.")

### Option C — Upload a zip
Zip the project locally (exclude `.venv`), upload, then extract.

Include `data/verify_docs_sft_500.jsonl` in the zip.

In [ ]:
from pathlib import Path

USE_ZIP_UPLOAD = False
ZIP_NAME = "SLM-Finetuning.zip"
PROJECT_DIR = Path("/content/SLM-Finetuning")

if USE_ZIP_UPLOAD:
    from google.colab import files
    import zipfile
    uploaded = files.upload()  # choose your zip
    zpath = Path(list(uploaded.keys())[0])
    PROJECT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zpath, "r") as zf:
        zf.extractall(PROJECT_DIR)
    # if zip contains a single top folder, enter it
    kids = [p for p in PROJECT_DIR.iterdir() if p.is_dir() and (p / "pyproject.toml").exists()]
    if kids:
        PROJECT_DIR = kids[0]
    %cd {PROJECT_DIR}
    print("Extracted to", PROJECT_DIR)
else:
    print("Zip upload skipped.")

## 3) Install dependencies (Unsloth CUDA)

In [ ]:
import os
from pathlib import Path

# Ensure we are inside the project
if Path("pyproject.toml").exists():
    PROJECT_DIR = Path(".").resolve()
elif "PROJECT_DIR" in globals():
    %cd {PROJECT_DIR}
else:
    raise SystemExit("Project not found. Run Option A/B/C first.")

print("Project:", Path(".").resolve())

# Unsloth recommended Colab install
!pip install -q --upgrade pip
!pip install -q unsloth
!pip install -q -e .

print("Install done.")

## 4) Prepare motor verify-docs dataset

Builds `data/processed/train.jsonl` (400) + `eval.jsonl` (50) and held-out eval (50 demo) from `data/verify_docs_sft_500.jsonl`.
Uses the **sharegpt** (`messages`) template — not the credit-card Alpaca set.

In [ ]:
import json
from pathlib import Path

src = Path("data/verify_docs_sft_500.jsonl")
assert src.exists(), f"Missing {src.resolve()} — upload the project with this file."

rows = [json.loads(l) for l in src.read_text(encoding="utf-8").splitlines() if l.strip()]
train = [r for r in rows if r.get("split") == "train"]
val = [r for r in rows if r.get("split") == "val"]
demo = [r for r in rows if r.get("split") == "demo"]
print(f"Loaded {len(rows)} rows → train={len(train)} val={len(val)} demo={len(demo)}")

Path("data/processed").mkdir(parents=True, exist_ok=True)
Path("data/evaluation").mkdir(parents=True, exist_ok=True)

def write_jsonl(path: Path, items: list):
    path.write_text("\n".join(json.dumps(r, ensure_ascii=False) for r in items) + "\n", encoding="utf-8")

write_jsonl(Path("data/processed/train.jsonl"), train)
write_jsonl(Path("data/processed/eval.jsonl"), val)

held = []
for r in demo:
    msgs = r.get("messages") or []
    sys = next((m["content"] for m in msgs if m.get("role") == "system"), "")
    user = next((m["content"] for m in msgs if m.get("role") == "user"), "")
    asst = next((m["content"] for m in msgs if m.get("role") == "assistant"), "")
    prompt = (
        f"<|im_start|>system\n{sys}<|im_end|>\n"
        f"<|im_start|>user\n{user}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )
    held.append({
        "id": r.get("id"),
        "claim_id": r.get("claim_id"),
        "hard_case": r.get("hard_case"),
        "system": sys,
        "prompt": prompt,
        "expected": asst,
        "labels": r.get("labels"),
        "split": "heldout",
        "notes": "Motor OD verify-docs demo split; not used for fine-tuning.",
    })

write_jsonl(Path("data/evaluation/heldout_comparison_eval.jsonl"), held)
write_jsonl(Path("data/evaluation/usecase_eval.jsonl"), held)

!wc -l data/verify_docs_sft_500.jsonl data/processed/train.jsonl data/processed/eval.jsonl data/evaluation/heldout_comparison_eval.jsonl

# Sanity: sharegpt formatter keeps system + user + assistant
from slm_finetune.data.dataset import format_sharegpt
sample = format_sharegpt(train[0])["text"]
assert "<|im_start|>system" in sample and "<|im_start|>assistant" in sample
print("sharegpt sample chars:", len(sample))
print("OK — motor verify-docs data ready.")

## 5) Train with Colab configs (SFT + QLoRA)

- Model: `configs/model/colab.yaml` (`max_seq_length: 1536` for claim packs)
- Training: `configs/training/colab.yaml` (3 epochs on 400 rows)
- Data: `configs/data/default.yaml` (`instruction_template: sharegpt`)

In [ ]:
import os
os.environ.setdefault("WANDB_DISABLED", "true")

!slm train \
  --method qlora \
  --model-config configs/model/colab.yaml \
  --training-config configs/training/colab.yaml \
  --data-config configs/data/default.yaml \
  --run-name motor-verify-docs-colab

## 6) Metrics

In [ ]:
!slm metrics list-runs
!slm metrics latest

### Use-case eval (JSON extraction on held-out demo)

Runs generation on the 50 held-out motor claim packs and logs use-case KPIs.

In [ ]:
from pathlib import Path

RUN_ID = "motor-verify-docs-colab"
adapter = Path("models/finetuned") / RUN_ID
assert adapter.exists(), f"Adapter not found: {adapter} — check models/finetuned/"

!slm eval-usecase \
  --model-dir {adapter} \
  --run-id {RUN_ID} \
  --usecase-file data/evaluation/usecase_eval.jsonl \
  --data-config configs/data/default.yaml

### Optional: list adapters / quick smoke prompt

After training finishes, confirm the adapter folder exists. The credit-card intent comparison report is **not** used for this use case.

In [ ]:
from pathlib import Path

ft_dirs = sorted(Path("models/finetuned").glob("*"))
ft_dirs = [p for p in ft_dirs if p.is_dir() and p.name != "README.md"]
print("Available adapters:")
for p in ft_dirs:
    print(" -", p.name)

RUN_ID = "motor-verify-docs-colab"
adapter = Path("models/finetuned") / RUN_ID
print("Primary adapter:", adapter, "exists=" + str(adapter.exists()))

## 7) Download artifacts to your laptop

In [ ]:
from pathlib import Path
import shutil
from google.colab import files

RUN_ID = "motor-verify-docs-colab"
bundle = Path("/content/slm_finetune_artifacts")
if bundle.exists():
    shutil.rmtree(bundle)
bundle.mkdir(parents=True)

for src in [
    Path("models/finetuned") / RUN_ID,
    Path("experiments") / RUN_ID,
    Path("artifacts/reports") / RUN_ID,
    Path("artifacts/metrics/metrics.db"),
]:
    if not src.exists():
        print("skip missing", src)
        continue
    dest = bundle / src.name
    if src.is_dir():
        shutil.copytree(src, dest)
    else:
        shutil.copy2(src, dest)

zip_path = shutil.make_archive(str(bundle), "zip", root_dir=bundle)
print("Created", zip_path)
files.download(zip_path)

## Notes

- **Task:** motor OD document verification → compact JSON (not credit-card intents).
- **Method:** SFT with QLoRA (`--method qlora`).
- **Template:** `sharegpt` / `messages` (system + user + assistant).
- Local Mac uses `configs/training/default.yaml` (MLX).
- Colab uses `configs/training/colab.yaml` + `configs/model/colab.yaml` (CUDA).
- Free Colab sessions can disconnect; save adapters to Drive if runs are long.
- After download, copy the adapter into your laptop `models/finetuned/<run_id>/`.